In [6]:

# 1. Import Libraries

import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 2. Download NLTK Resources


nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')



# 3. Load IMDb Dataset


df = pd.read_csv("IMDB Dataset.csv")

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print()


# 4. Display First 5 Rows

print("First 5 rows:")
print(df.head())
print()


# 5. Check Missing Values

print("Missing values:")
print(df.isnull().sum())
print()


# 6. Check Duplicate Reviews

duplicates = df['review'].duplicated().sum()

print("Number of duplicate reviews:", duplicates)
print()


# 7. Remove Duplicate Reviews

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))
print()


# 8. Prepare Stopwords and Lemmatizer

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()


# 9. Create Cleaning Function

def clean_review(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = nltk.word_tokenize(text)

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Join words
    cleaned_text = ' '.join(words)

    return cleaned_text


# 10. Clean All Reviews

print("Cleaning reviews...")

df['cleaned_review'] = df['review'].apply(clean_review)

print("Reviews cleaned successfully!")
print()


# 11. Display Original and Cleaned Reviews


print("============================================")
print("ORIGINAL VS CLEANED REVIEWS")
print("============================================")

for i in range(5):

    print("\nOriginal Review:")
    print(df['review'].iloc[i])

    print("\nCleaned Review:")
    print(df['cleaned_review'].iloc[i])

    print("\nSentiment:")
    print(df['sentiment'].iloc[i])

    print("--------------------------------------------")


# 12. Convert Sentiment into Numbers

df['sentiment_label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})


# 13. Separate Input and Output

X = df['cleaned_review']

y = df['sentiment_label']


# 14. Split Dataset into Training and Testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining reviews:", len(X_train))
print("Testing reviews:", len(X_test))


# 15. Convert Text into TF-IDF

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)


# 16. Train Logistic Regression Model

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("\nModel training completed!")


# 17. Predict Sentiment

y_pred = model.predict(X_test_tfidf)


# 18. Calculate Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("MODEL RESULTS")
print("============================================")

print("Accuracy:", accuracy)


# 19. Classification Report


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=['negative', 'positive']
    )
)


# 20. Confusion Matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# 21. Predict New Reviews

new_reviews = [
    "This movie was amazing and I really loved it",
    "This movie was boring and terrible",
    "Excellent movie with great acting",
    "Worst movie I have ever watched"
]


# Clean new reviews
cleaned_new_reviews = [
    clean_review(review)
    for review in new_reviews
]


# Convert new reviews to TF-IDF
new_reviews_tfidf = vectorizer.transform(
    cleaned_new_reviews
)


# Predict
predictions = model.predict(
    new_reviews_tfidf
)

# 22. Display Predictions

print("\n============================================")
print("NEW REVIEW PREDICTIONS")
print("============================================")

for review, cleaned, prediction in zip(
    new_reviews,
    cleaned_new_reviews,
    predictions
):

    if prediction == 1:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("\nOriginal Review:")
    print(review)

    print("Cleaned Review:")
    print(cleaned)

    print("Predicted Sentiment:")
    print(sentiment)

    print("--------------------------------------------")


# 23. Add Predictions to Test Dataset

result_df = pd.DataFrame({
    'review': X_test.values,
    'actual_sentiment': y_test.values,
    'predicted_sentiment': y_pred
})


# Convert numbers back to words

result_df['actual_sentiment'] = result_df[
    'actual_sentiment'
].map({
    1: 'positive',
    0: 'negative'
})

result_df['predicted_sentiment'] = result_df[
    'predicted_sentiment'
].map({
    1: 'positive',
    0: 'negative'
})


# 24. Save Cleaned Dataset

df.to_csv(
    "IMDB_cleaned.csv",
    index=False
)


# 25. Save Prediction Dataset

result_df.to_csv(
    "IMDB_predictions.csv",
    index=False
)



# 26. Final Output

print("\n============================================")
print("PROCESS COMPLETED SUCCESSFULLY!")
print("============================================")

print("\nCleaned dataset:")
print("IMDB_cleaned.csv")

print("\nPrediction dataset:")
print("IMDB_predictions.csv")

print("\nTotal reviews:", len(df))

print("\nFinal columns:")
print(df.columns.tolist())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Divya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Dataset loaded successfully!
Number of rows: 49396

First 5 rows:
                                              review sentiment
0  $25,000 Pyramid Clues: Deep Blue Sea. Tremors....  negative
1  0.5/10. This movie has absolutely nothing good...  negative
2  0*'s Christian Slater, Tara Reid, Stephen Dorf...  negative
3  102 Dalmatians (2000, Dir. Kevin Lima) <br /><...  negative
4  102 DALMATIANS [Walt Disney]: I wasn't a fan o...  negative

Missing values:
review       0
sentiment    0
dtype: int64

Number of duplicate reviews: 0

Rows after removing duplicates: 49396

Cleaning reviews...
Reviews cleaned successfully!

ORIGINAL VS CLEANED REVIEWS

Original Review:
$25,000 Pyramid Clues: Deep Blue Sea. Tremors. Slither. Eight Legged Freaks.<br /><br />Pyramid Category: Movies that were funnier and more thrilling than Snakes on a Plane.<br /><br />Hell, with that definition I'd have to include the relatively harrowing journey of Ted and Elaine in Airplane! as superior to Snakes in both l